# Duplicate Leakage Analysis

## Objective

Investigate whether identical feature combinations with different target values affect model performance.

The dataset contains duplicate feature rows where multiple properties have identical input features but different prices.

We will compare model performance before and after removing these duplicate feature combinations.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/house_prices_refined.csv")

In [2]:
feature_columns = [
    "bhk",
    "propertytype",
    "location",
    "sqft"
]

In [3]:
df.duplicated(
    subset=feature_columns
).sum()

np.int64(3938)

In [4]:
duplicate_groups = (
    df.groupby(feature_columns)
      .size()
      .reset_index(name="count")
)

duplicate_groups.head()

,bhk,propertytype,location,sqft,count
0,1,Flat,Ahmedabad,422,1
1,1,Flat,Ahmedabad,441,1
2,1,Flat,Ahmedabad,486,1
3,1,Flat,Ahmedabad,580,1
4,1,Flat,Ahmedabad,585,1


In [5]:
duplicate_groups = (
    df.groupby(feature_columns)
      .agg(
          row_count=("totalprice", "size"),
          unique_prices=("totalprice", "nunique"),
          min_price=("totalprice", "min"),
          max_price=("totalprice", "max")
      )
      .reset_index()
)

conflicting_groups = duplicate_groups[
    duplicate_groups["unique_prices"] > 1
]

conflicting_groups.sort_values(
    "row_count",
    ascending=False
).head(20)

,bhk,propertytype,location,sqft,row_count,unique_prices,min_price,max_price
6223,3,Flat,NewDelhi,1500,24,24,12000000,105000000
6243,3,Flat,NewDelhi,1800,23,23,15000000,92500000
4475,3,Flat,Dwarka,1600,23,23,4400000,41000000
4471,3,Flat,Dwarka,1500,23,23,1700000,29000000
4486,3,Flat,Dwarka,1800,19,19,3200000,35000000
6229,3,Flat,NewDelhi,1600,17,17,21000000,60000000
6218,3,Flat,NewDelhi,1400,17,17,7300000,72500000
6209,3,Flat,NewDelhi,1200,16,16,5260000,40000000
4481,3,Flat,Dwarka,1700,16,16,1300000,37000000
6205,3,Flat,NewDelhi,1100,15,15,5700000,30000000


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_keys = set(
    train_df[feature_columns].astype(str).agg("|".join, axis=1)
)

test_keys = set(
    test_df[feature_columns].astype(str).agg("|".join, axis=1)
)

overlapping_keys = train_keys.intersection(test_keys)

print("Unique feature groups in train:", len(train_keys))
print("Unique feature groups in test:", len(test_keys))
print("Feature groups appearing in both:", len(overlapping_keys))
print(
    "Percentage of test feature groups overlapping:",
    round(len(overlapping_keys) / len(test_keys) * 100, 2),
    "%"
)

Unique feature groups in train: 7789
Unique feature groups in test: 2363
Feature groups appearing in both: 793
Percentage of test feature groups overlapping: 33.56 %
